# EXPLAIN — common notebook (adapter-driven)

One model-agnostic notebook to **diagnose and explain** a trained model. The
per-model logic lives in an *explain adapter* (`CLASSIFIER/adapters/explain.py`)
selected by `ADAPTER` (defaults to `MODEL`); the SHARED cells below call only the
explain-adapter contract, so they are identical across models. Genuinely
model-specific visuals run in **capability-guarded "extra" cells** that no-op when
the active adapter does not declare them.

* `gaae`  — latent space, GAT attention, reconstruction, latent-dim IG / GNNExplainer.
* `gec`   — flattened-trajectory MLP: diagnostics, per-visit trajectories, early
  detection, latent/visit attribution.
* `gelstm` / `gegru` — recurrent: the above plus hidden-state trajectory, visit
  occlusion, and Δt / order temporal ablations.

Classifiers reload their latest trained run (`source_experiment`); GAAE loads its
encoder checkpoint. Driven by `run_experiment.py` via `the experiments/ directory`; also
runnable standalone (interactive prompts preserved).

In [1]:
# === Papermill parameters (injected by run_experiment.py) ===
EXPERIMENT_ID = None
MODE = None
MODEL = None
ADAPTER = None                # explain-adapter key; None -> defaults to MODEL
DATASET = None
SEED = 42
GAAE_CHECKPOINT_PATH = None   # GAAE encoder (gaae adapter); None -> interactive pick
SOURCE_EXPERIMENT = None      # classifiers reload outputs/<id>/latest/
THRESHOLD_MODE = None         # unused here (reload uses the saved threshold)
FIXED_THRESHOLD = None
WANDB_ENABLED = True
OUTPUT_DIR = None
RESOLVED_CONFIG = None        # merged hyperparameter dict
RUN_DIR = None                # set by the runner: where figures / summary go
RUN_NAME = None               # set by the runner

In [2]:
# Parameters
EXPERIMENT_ID = "explain-vgae-gcn"
MODE = "explain"
MODEL = "VGAE"
DATASET = "DELCODE_WHOLE_BRAIN"
SEED = 42
GAAE_CHECKPOINT_PATH = "notebooks/checkpoints/checkpoints_vgae_whole_brain/VGAE_GCN.pth"
THRESHOLD_MODE = None
FIXED_THRESHOLD = None
WANDB_ENABLED = True
OUTPUT_DIR = "outputs/explain-vgae-gcn"
RESOLVED_CONFIG = {"seed": 100, "batch_size": 64, "learning_rate": 0.001, "weight_decay": 0.001, "beta": 1.0, "epochs": 500, "early_stopping_patience": 25, "latent_dim": 64, "hidden_dim": 128, "conv_type": "gcn", "num_heads": 2, "dropout": 0.3, "adjacency_k": 16, "num_workers": 8, "file_variant": "z_transformed"}
RUN_DIR = "/mnt/e/fyassine/ad-early-detection/CLASSIFIER/outputs/explain-vgae-gcn/runs/ancient-brook-11-bd6a29e3c-2026-06-23_21-33-14"
RUN_NAME = "ancient-brook-11-bd6a29e3c-2026-06-23_21-33-14"
ADAPTER = "vgae"


## Pipeline overview

`load` (HOOK) → `prepare_bundles` (HOOK) → latent space → diagnostics →
**data journey** (one subject: raw scan → FC → graph → layers → probability) →
region importance ("why") → capability-guarded extras → save figures + summary.

In [3]:
import sys
from pathlib import Path
repo_root = Path('/mnt/e/fyassine/ad-early-detection')
model_root = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(model_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
    sys.path.insert(0, str(model_root))

In [4]:
# reproducibility seeding — must run before datasets / models.
from CLASSIFIER.common.seeding import set_seed, make_rng, make_torch_generator
set_seed(SEED)
rng = make_rng(SEED)
torch_gen = make_torch_generator(SEED)

In [5]:
import json, os, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from common.checkpoints import select_gaae_checkpoint
from common.sanity import run_full_audit
from common.provenance import region_from_data_root
from common.crossval import Bundle
from common import tracking
from common import explain as ce
from common import pipeline_trace as pt
from adapters.explain import get_explain_adapter

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Imports OK  |  device:', device)

Imports OK  |  device: cuda


## Configuration

In [6]:
from DATA.src.splitting.load_splits import splits_dir, split_csv_paths

WB_DATA_ROOT = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices'
METADATA_DIR = '/mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata'
COHORTS_CSV  = os.path.join(METADATA_DIR, 'cohorts.csv')
SPLITS_DIR   = str(splits_dir('downstream'))
TRAIN_CSV = os.path.join(SPLITS_DIR, 'train.csv')
VAL_CSV   = os.path.join(SPLITS_DIR, 'val.csv')
TEST_CSV  = os.path.join(SPLITS_DIR, 'test.csv')

DATA_INFO = region_from_data_root(WB_DATA_ROOT)
REGION = DATA_INFO['region']
print(f"Input data: region={REGION}  atlas={DATA_INFO['atlas']}")

CHECKPOINT_SEARCH_DIRS = [str(model_root / 'notebooks' / 'checkpoints' / 'checkpoints_gaae_whole_brain')]
TRAIN_CONFIG = dict(RESOLVED_CONFIG) if RESOLVED_CONFIG else {}

# GAAE encoder arch (must match the checkpoint).
CONFIG_PATH = model_root / 'configs' / 'gaae_delcode_whole_brain.json'
if CONFIG_PATH.exists():
    hp = json.loads(CONFIG_PATH.read_text())
else:
    hp = dict(latent_dim=64, hidden_dim=128, num_heads=2, cond_dim=2, dropout=0.3,
              adjacency_k=8, file_variant='z_transformed')
print('GAAE config:', hp)

# Where figures / explain_summary.json go.
OUT_DIR = Path(RUN_DIR) if RUN_DIR else (model_root / 'outputs' / (EXPERIMENT_ID or 'explain') / 'standalone')
OUT_DIR.mkdir(parents=True, exist_ok=True)
ATLAS = ce.load_schaefer_atlas()
SUMMARY = {'experiment_id': EXPERIMENT_ID, 'adapter': (ADAPTER or MODEL), 'figures': []}
print('Output dir:', OUT_DIR)

Input data: region=wholebrain  atlas=sch200
GAAE config: {'seed': 100, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 0.001, 'adj_loss_weight': 0.2, 'epochs': 500, 'early_stopping_patience': 25, 'latent_dim': 64, 'hidden_dim': 128, 'num_heads': 2, 'cond_dim': 2, 'dropout': 0.3, 'adjacency_k': 16, 'num_workers': 8, 'file_variant': 'z_transformed'}
Output dir: /mnt/e/fyassine/ad-early-detection/CLASSIFIER/outputs/explain-vgae-gcn/runs/ancient-brook-11-bd6a29e3c-2026-06-23_21-33-14


In [7]:
# split-hygiene audit — hard-fails if any subject crosses splits.
_ = run_full_audit(split_csv_paths('downstream'))

[SANITY] Split sizes: {'train': 99, 'val': 34, 'test': 34}
[SANITY] Pairwise-disjoint: OK


## Resolve adapter & load the trained model

`get_explain_adapter(ADAPTER or MODEL)` returns the explain adapter. `load()` builds
the GAAE encoder (gaae) or reloads `outputs/<SOURCE_EXPERIMENT>/latest/` (classifiers).

In [8]:
ADAPTER_KEY = (ADAPTER or MODEL or '').lower()
# Encoder adapters (gaae / vgae) load a checkpoint directly; classifiers reload a
# trained run via SOURCE_EXPERIMENT instead.
ENCODER_ADAPTERS = {'gaae', 'vgae'}
IS_ENCODER = ADAPTER_KEY in ENCODER_ADAPTERS

GAAE_CKPT = None
if IS_ENCODER:
    if GAAE_CHECKPOINT_PATH is None and RUN_DIR is not None:
        raise ValueError(f"GAAE_CHECKPOINT_PATH (checkpoint_path) is required under the "
                         f"runner for the {ADAPTER_KEY!r} adapter.")
    if ADAPTER_KEY == 'gaae':
        _gaae_name, _gaae_ckpt, _gaae_dir = select_gaae_checkpoint(
            CHECKPOINT_SEARCH_DIRS, checkpoint_path=GAAE_CHECKPOINT_PATH)
        GAAE_CKPT = str(_gaae_ckpt)
        print('GAAE checkpoint:', _gaae_name)
    else:  # vgae: its checkpoints live in their own dir; the path is supplied directly.
        if GAAE_CHECKPOINT_PATH is None:
            raise ValueError("vgae explain needs checkpoint_path to a model.VGAE checkpoint "
                             "(set 'checkpoint_path:' on the the experiments/ directory entry).")
        GAAE_CKPT = str(GAAE_CHECKPOINT_PATH if os.path.isabs(str(GAAE_CHECKPOINT_PATH))
                        else model_root / GAAE_CHECKPOINT_PATH)
        print('VGAE checkpoint:', GAAE_CKPT)

adapter = get_explain_adapter(ADAPTER_KEY)(
    gaae_ckpt_path=GAAE_CKPT, gaae_hp=hp, train_config=TRAIN_CONFIG,
    data_root=WB_DATA_ROOT, cohorts_csv=COHORTS_CSV, device=device, rng=rng,
    source_experiment=SOURCE_EXPERIMENT, classifier_root=model_root)
CAPS = adapter.capabilities
CTX_INFO = adapter.load()
print(f'Adapter: {type(adapter).__name__}  capabilities={sorted(CAPS)}')
print('load():', json.dumps({k: str(v) for k, v in CTX_INFO.items()}, indent=2))

VGAE checkpoint: /mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints/checkpoints_vgae_whole_brain/VGAE_GCN.pth


Adapter: VGAEExplainAdapter  capabilities=['attention', 'reconstruction']
load(): {
  "encoder": "VariationalGraphAutoencoder",
  "conv_type": "gcn",
  "vgae_checkpoint": "/mnt/e/fyassine/ad-early-detection/CLASSIFIER/notebooks/checkpoints/checkpoints_vgae_whole_brain/VGAE_GCN.pth"
}


In [9]:
# Optional W&B run (figures can be logged; off by default unless WANDB_ENABLED).
_wb_exp = {'id': EXPERIMENT_ID or 'explain', 'mode': MODE or 'explain',
           'model': MODEL or ADAPTER_KEY, 'dataset': DATASET or REGION,
           'seed': SEED, 'wandb': WANDB_ENABLED}
WANDB_RUN = tracking.init_run(_wb_exp, {**TRAIN_CONFIG, 'REGION': REGION, 'adapter': ADAPTER_KEY})

In [10]:
train_df = pd.read_csv(TRAIN_CSV); val_df = pd.read_csv(VAL_CSV); test_df = pd.read_csv(TEST_CSV)
cv_pool_df = pd.concat([train_df, val_df], ignore_index=True)
CV_BUNDLE, TEST_BUNDLE = adapter.prepare_bundles(cv_pool_df, test_df)
ALL_ITEMS = list(CV_BUNDLE.items) + list(TEST_BUNDLE.items)
ALL_BUNDLE = Bundle([it['label'] for it in ALL_ITEMS],
                    [it['subject_id'] for it in ALL_ITEMS], ALL_ITEMS)
print(f'CV subjects: {len(CV_BUNDLE)}  Test subjects: {len(TEST_BUNDLE)}  All: {len(ALL_BUNDLE)}')

LongitudinalSubjectDataset[v2]: 133 subjects (54 converter, 79 stable MCI)
  Scans per subject: min=1  max=6  mean=3.0


LongitudinalSubjectDataset[v2]: 34 subjects (14 converter, 20 stable MCI)
  Scans per subject: min=1  max=6  mean=2.9


CV subjects: 133  Test subjects: 34  All: 167


## Latent space

In [11]:
X, LBL, SIDS = adapter.latent_embeddings(ALL_BUNDLE)
method = 'umap' if X.shape[0] >= 8 else 'pca'
EMB2D = ce.embed_2d(X, method, seed=SEED)
fig = ce.plot_latent_space(EMB2D, LBL, method_name=method.upper(),
                           title=f'Latent space ({method.upper()})  |  {type(adapter).__name__}')
plt.show()
SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'latent_space')]

## Diagnostics

In [ ]:
DIAG = adapter.diagnostics(TEST_BUNDLE)
if DIAG.get('kind') == 'classification':
    print(f"Reloaded test AUC = {DIAG['reloaded_auc']:.4f}   (saved: {DIAG.get('saved_auc')})")
    print(f"sens={DIAG['sensitivity']:.3f}  spec={DIAG['specificity']:.3f}  "
          f"f1={DIAG['f1']:.3f}  ECE={DIAG['ece']:.3f}  thr={DIAG['threshold']:.3f}")
    fig = ce.plot_classification_diagnostics(DIAG['targets'], DIAG['probs'], DIAG['threshold'],
                                             title=f"Test diagnostics  |  {type(adapter).__name__}")
    SUMMARY['diagnostics'] = {k: DIAG[k] for k in ('auc','sensitivity','specificity','f1','ece','threshold')}
    SUMMARY['diagnostics']['saved_auc'] = DIAG.get('saved_auc')
else:
    err, lab = DIAG['recon_error'], DIAG['labels']
    rfid = np.asarray(DIAG['recon_fidelity_r'], dtype=float)
    med, (q1, q3) = DIAG['fidelity_median'], DIAG['fidelity_iqr']
    print(f"Reconstruction error  converter={DIAG['mean_converter']:.4f}  "
          f"stable={DIAG['mean_stable']:.4f}  one-vs-rest AUC={DIAG['recon_error_auc']:.3f}")
    print(f"Cohort fidelity (input↔recon Pearson r):  median={med:.3f}  "
          f"IQR=[{q1:.3f}, {q3:.3f}]  (n={rfid.size} test subjects)")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    # left: reconstruction error by cohort
    for l, c, nm in [(1, '#F44336', 'converter'), (0, '#2196F3', 'stable MCI')]:
        axes[0].hist(err[lab == l], bins=20, alpha=0.6, color=c, label=nm)
    axes[0].set_xlabel('mean per-ROI reconstruction error (MSE)'); axes[0].set_ylabel('subjects')
    axes[0].legend(); axes[0].set_title('GAAE reconstruction error by cohort'); axes[0].grid(alpha=0.3)
    # right: cohort reconstruction-fidelity distribution with median/IQR + reference bands
    axes[1].hist(rfid, bins=20, color='#4682B4', alpha=0.85)
    axes[1].axvspan(q1, q3, color='black', alpha=0.10, label=f'IQR [{q1:.3f}, {q3:.3f}]')
    axes[1].axvline(med, color='black', lw=1.6, label=f'median r={med:.3f}')
    for thr in (0.60, 0.80, 0.90):
        axes[1].axvline(thr, color='grey', ls='--', lw=0.9)
    axes[1].set_xlabel('input↔reconstruction Pearson r'); axes[1].set_ylabel('subjects')
    axes[1].set_title('Cohort reconstruction fidelity (band lines: 0.60/0.80/0.90)')
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    SUMMARY['diagnostics'] = {k: DIAG[k] for k in
                              ('recon_error_auc', 'mean_converter', 'mean_stable',
                               'fidelity_median', 'fidelity_iqr')}
plt.show()
SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'diagnostics')]

## Data journey — one subject, raw scan → prediction

The subject with the **most visits** is traced end-to-end. Each stage prints its
tensor shape so the transformation `(200,200) → … → P` is explicit. If the raw
`.nii` is missing, the trace falls back to the stored FC matrix.

In [ ]:
SUBJECT = adapter.pick_walkthrough_subject(TEST_BUNDLE if len(TEST_BUNDLE) else ALL_BUNDLE)
BASE_VISIT = adapter.baseline_visit(SUBJECT)
print(f"Walkthrough subject: {SUBJECT['subject_id']}  "
      f"label={'converter' if SUBJECT['label'] else 'stable_mci'}  "
      f"n_visits={SUBJECT.get('n_scans')}  baseline=M{BASE_VISIT}")
TRACE_RAW = pt.nii_to_fc_to_graph(SUBJECT['subject_id'], BASE_VISIT,
                                  file_variant=hp.get('file_variant', 'z_transformed'),
                                  adjacency_k=hp.get('adjacency_k', 8))
print('source:', TRACE_RAW['source'])

In [ ]:
# Stage 1-2: raw BOLD volume + Schaefer-200 timeseries.
if TRACE_RAW.get('bold_img') is not None:
    fig = pt.plot_brain_slice(TRACE_RAW['bold_img'], title=f"{SUBJECT['subject_id']} — mean BOLD (M{BASE_VISIT})")
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_1_brain')]
    ts = TRACE_RAW['timeseries']
    print('timeseries shape:', ts.shape)
    fig, ax = plt.subplots(figsize=(11, 3.5))
    for r in range(min(6, ts.shape[1])):
        ax.plot(ts[:, r], lw=0.8, alpha=0.8, label=f'ROI {r}')
    ax.set_xlabel('TR'); ax.set_ylabel('z-BOLD'); ax.set_title('Schaefer-200 timeseries (first 6 ROIs)')
    ax.legend(ncol=6, fontsize=7); ax.grid(alpha=0.3)
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_2_timeseries')]
else:
    print('[skip] raw .nii unavailable — showing stored FC matrix only (stages 1-2 skipped).')

In [ ]:
# Stage 3: functional-connectivity matrix.
fig = pt.plot_fc_heatmap(TRACE_RAW['fc_matrix'], TRACE_RAW.get('z_fc'))
print('FC matrix shape:', np.asarray(TRACE_RAW['x']).shape)
plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_3_fc')]

In [ ]:
# Stage 4: kNN brain graph.
print('edge_index shape:', TRACE_RAW['edge_index'].shape)
fig = pt.plot_brain_graph(TRACE_RAW['edge_index'], ATLAS, title=f"kNN brain graph — {SUBJECT['subject_id']}")
plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_4_graph')]

In [ ]:
# Stage 5: through the model layers (adapter-specific intermediates).
TRACE = adapter.trace_forward(SUBJECT)
print('Forward stages:')
for nm, shp in TRACE['stages']:
    print(f'  {nm:30s} {tuple(shp)}')

if 'decoder_gat1' in TRACE and 'recon' in TRACE:
    # Autoencoder (GAAE): row 1 = encoding -> latent (bottleneck), row 2 = latent -> decoding.
    layout = {
        (0, 0): ('x', 'Encoder entry (input x)'),
        (0, 1): ('gat2', 'Encoder pre-latent (gat2)'),
        (0, 2): ('latent', 'Latent z (bottleneck)'),
        (1, 2): ('decoder_gat1', 'Decoder post-latent (decoder_gat1)'),
        (1, 3): ('recon', 'Reconstruction x̂'),
    }
    fig, axes = plt.subplots(2, 4, figsize=(20, 8.4))
    for ax in axes.flat:
        ax.axis('off')
    for (r, c), (key, title) in layout.items():
        ax = axes[r, c]; ax.axis('on')
        M = np.asarray(TRACE[key])
        im = ax.imshow(M, aspect='auto', cmap='viridis')
        ax.set_title(f'{title}\n{M.shape}', fontsize=10)
        ax.set_xlabel('feature'); ax.set_ylabel('node')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'Encode → latent → decode  |  {type(adapter).__name__}')
    fig.tight_layout()
    plt.show()
    SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_5_layers')]
else:
    panels = [(k, TRACE[k]) for k in ('gat1', 'gat2', 'latent', 'visit_embeddings', 'hidden_states')
              if k in TRACE]
    if panels:
        fig, axes = plt.subplots(1, len(panels), figsize=(5.0 * len(panels), 4.2))
        if len(panels) == 1: axes = [axes]
        for ax, (nm, M) in zip(axes, panels):
            M = np.asarray(M)
            im = ax.imshow(M, aspect='auto', cmap='viridis')
            ax.set_title(f'{nm}  {M.shape}'); ax.set_xlabel('feature'); ax.set_ylabel('node / visit')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.tight_layout(); plt.show()
        SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_5_layers')]

In [ ]:
# Stage 5b: input vs reconstruction (200x200) + residual, with a fidelity score.
if 'recon' in TRACE:
    from model.GAAE.explain import reconstruction_quality

    Q = reconstruction_quality(TRACE['x'], TRACE['recon'])
    X_in = np.asarray(TRACE['x']); X_rec = np.asarray(TRACE['recon'])
    vmax = float(np.nanpercentile(np.abs(X_in), 99)) or 1.0          # shared scale: input vs recon
    rmax = float(np.nanpercentile(np.abs(Q['residual']), 99)) or 1.0  # residual on its own scale

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))
    panels = [
        (axes[0], Q['residual'], -rmax, rmax, f"Residual (x̂ − x)  {Q['residual'].shape}"),
        (axes[1], X_in,          -vmax, vmax, f"Input x  {X_in.shape}"),
        (axes[2], X_rec,         -vmax, vmax, f"Reconstruction x̂  {X_rec.shape}"),
    ]
    for ax, M, vlo, vhi, title in panels:
        im = ax.imshow(M, cmap='RdBu_r', vmin=vlo, vmax=vhi, aspect='equal')
        ax.set_title(title); ax.set_xlabel('ROI'); ax.set_ylabel('ROI')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(
        f"{SUBJECT['subject_id']}   |   r={Q['pearson_r']:.3f}   R²={Q['r2']:.3f}   "
        f"RMSE={Q['rmse']:.3f}   NRMSE={Q['nrmse']:.2f}   →   {Q['quality'].upper()}",
        fontsize=12,
    )
    fig.tight_layout()
    plt.show()
    SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_5b_recon_compare')]
    SUMMARY['reconstruction_fidelity'] = {
        k: Q[k] for k in ('mse', 'rmse', 'mae', 'nrmse', 'pearson_r', 'r2', 'quality')
    }
    print(
        "Reconstruction fidelity (decoder x̂ vs input x):\n"
        f"  Pearson r = {Q['pearson_r']:.3f}   R² = {Q['r2']:.3f}\n"
        f"  RMSE = {Q['rmse']:.4f}   MAE = {Q['mae']:.4f}   NRMSE = {Q['nrmse']:.3f}\n"
        f"  fidelity band: {Q['quality'].upper()}\n"
        "  Reference (FC-autoencoder rule-of-thumb): r≥0.90 excellent · 0.80–0.90 good · "
        "0.60–0.80 fair · <0.60 poor.\n"
        "  NRMSE≲0.5 means the residual is well under half the FC signal's own spread "
        "(NRMSE=√(1−R²) here); a near-uniform, low-amplitude residual map (no block "
        "structure left) indicates the network preserved the connectivity pattern."
    )
    # Cohort context — "good" relative to what this encoder typically achieves (from Diagnostics).
    _coh = np.asarray(DIAG.get('recon_fidelity_r', []), dtype=float)
    if _coh.size:
        _pct = float((_coh <= Q['pearson_r']).mean() * 100)
        print(f"  cohort context: this subject's r={Q['pearson_r']:.3f} vs test-set median "
              f"{DIAG['fidelity_median']:.3f}  (≈{_pct:.0f}th percentile of {_coh.size} subjects).")

In [ ]:
# Stage 6: prediction (classifiers) or reconstruction (GAAE / VGAE).
if 'probability' in CAPS:
    PRED = adapter.predict_one(SUBJECT)
    SUMMARY['walkthrough'] = {**PRED, 'subject_id': SUBJECT['subject_id']}
    fig, ax = plt.subplots(figsize=(8, 1.7))
    ax.barh([0], [PRED['prob']], color='#F44336' if PRED['pred'] else '#2196F3', height=0.5)
    ax.axvline(PRED['threshold'], color='black', ls='--', label=f"threshold={PRED['threshold']:.3f}")
    ax.set_xlim(0, 1); ax.set_yticks([])
    ax.set_title(f"P(converter)={PRED['prob']:.3f}  ->  "
                 f"pred={'converter' if PRED['pred'] else 'stable'}  "
                 f"(true={'converter' if PRED['true'] else 'stable'})")
    ax.legend(loc='lower right'); plt.show()
    SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'journey_6_prediction')]
    print(PRED)
else:
    # Visual input-vs-reconstruction comparison already shown in Stage 5b.
    # Adapter-agnostic: GAAE reconstructs features, VGAE reconstructs adjacency —
    # both expose the per-node error via the 'reconstruction' extra.
    err = adapter.extra('reconstruction', {'subject': SUBJECT})['per_node_error']
    print(f'mean per-ROI reconstruction error: {float(err.mean()):.4f}')

## Why — brain-region importance

Per-ROI importance for the walkthrough subject (GAT attention on the baseline graph,
always available; the extras below refine it with IG / GNNExplainer).

In [ ]:
REGION_IMP = adapter.region_importance(SUBJECT)
SUMMARY['region_importance_by_network'] = ce.network_importance_summary(REGION_IMP, ATLAS)
try:
    fig = ce.plot_region_importance_glassbrain(REGION_IMP, ATLAS,
            title=f'Region importance — {SUBJECT["subject_id"]}')
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'region_glassbrain')]
except Exception as exc:
    print('[glass-brain skipped]', exc)
fig = ce.plot_region_importance_bars(REGION_IMP, ATLAS,
        title=f'Region importance by network — {type(adapter).__name__}')
plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'region_bars')]

## Model-specific extras (capability-guarded)

Each cell runs only if the active adapter advertises that capability; otherwise it
prints `[skip]` and no-ops — so the single notebook serves every model.

In [ ]:
CTX = {'subject': SUBJECT, 'test_bundle': TEST_BUNDLE, 'latent_dim': 0}

def _maybe(name):
    if name in CAPS:
        return adapter.extra(name, CTX)
    print(f'[skip] {name!r} not supported by {type(adapter).__name__}')
    return None

In [ ]:
# GAAE: reconstruction error per ROI -> glass brain
res = _maybe('reconstruction')
if res is not None:
    err = res['per_node_error']
    fig = ce.plot_region_importance_bars(err, ATLAS, title='Reconstruction error by network')
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_reconstruction')]

In [ ]:
# GAAE: GNNExplainer node/edge importance for a latent dimension
res = _maybe('gnn_explainer') if 'gnn_explainer' in CAPS else _maybe('attention')
if res is not None and 'node_importance' in res:
    fig = ce.plot_region_importance_bars(res['node_importance'], ATLAS,
            title=f"GNNExplainer node importance (latent dim {res.get('latent_dim',0)})")
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_gnnexplainer')]

In [ ]:
# latent-dim / sequence integrated gradients (needs captum; degrades gracefully)
try:
    res = _maybe('latent_ig')
except ImportError as exc:
    res = None; print('[skip] latent_ig needs captum:', exc)
if res is not None:
    if 'per_visit' in res:   # classifiers: which visit / which latent dim
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].bar(range(len(res['per_visit'])), res['per_visit'], color='#4682B4')
        axes[0].set_title('IG importance per visit'); axes[0].set_xlabel('visit index')
        axes[1].bar(range(len(res['per_dim'])), res['per_dim'], color='#E69422')
        axes[1].set_title('IG importance per latent dim'); axes[1].set_xlabel('latent dim')
        fig.tight_layout(); plt.show()
        SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_latent_ig')]
    elif 'node_importance' in res:  # GAAE: per-ROI
        fig = ce.plot_region_importance_bars(res['node_importance'], ATLAS,
                title=f"Latent-dim {res.get('latent_dim',0)} integrated gradients")
        plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_latent_ig')]

In [ ]:
# classifiers: per-visit conversion-probability trajectory for the walkthrough subject
res = _maybe('trajectories')
if res is not None:
    pv = res['per_visit_probs']
    months = [m for m, _ in pv]; probs = [p for _, p in pv]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(months, probs, 'o-', color='#F44336')
    ax.axhline(adapter.threshold, ls='--', color='black', label=f'threshold={adapter.threshold:.3f}')
    ax.set_xlabel('visit month'); ax.set_ylabel('P(converter)'); ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"Conversion trajectory — {SUBJECT['subject_id']}"); ax.legend(); ax.grid(alpha=0.3)
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_trajectory')]

In [ ]:
# classifiers: early-detection AUC vs number of visits used
res = _maybe('early_detection')
if res is not None and res['early_detection']:
    rows = res['early_detection']
    ns = [r['n_visits'] for r in rows]; aucs = [r['auc'] for r in rows]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(ns, aucs, 'o-', color='#00760E')
    ax.set_xlabel('visits used (N)'); ax.set_ylabel('test AUC'); ax.set_ylim(0, 1.02)
    ax.set_title('Early detection — AUC vs visits'); ax.grid(alpha=0.3)
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_early_detection')]
    SUMMARY['early_detection'] = rows

In [ ]:
# GELSTM/GRU: RNN hidden-state trajectory across visits
res = _maybe('hidden_state')
if res is not None:
    H = np.asarray(res['hidden_states'])
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(H.T, aspect='auto', cmap='coolwarm')
    ax.set_xlabel('visit index'); ax.set_ylabel('hidden unit')
    ax.set_title(f'RNN hidden-state trajectory  {H.shape}')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_hidden_state')]

In [ ]:
# GELSTM/GRU: visit-occlusion importance (drop each visit, measure Δ probability)
res = _maybe('visit_occlusion')
if res is not None and res['occlusion']:
    occ = res['occlusion']
    months = [o['dropped_visit_month'] for o in occ]; deltas = [o['delta'] for o in occ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([f'M{m}' for m in months], deltas, color='#C43AFA')
    ax.set_xlabel('dropped visit'); ax.set_ylabel('Δ P(converter) when removed')
    ax.set_title('Visit occlusion importance'); ax.grid(alpha=0.3, axis='y')
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_visit_occlusion')]
    SUMMARY['visit_occlusion'] = occ

In [ ]:
# GELSTM/GRU: temporal ablations (zero Δt, shuffle visit order) — test-set AUC
res = _maybe('temporal_ablation')
if res is not None:
    ab = res['ablation']
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['full', 'zero Δt', 'shuffled'],
           [ab['auc_full'], ab['auc_zero_dt'], ab['auc_shuffled']],
           color=['#00760E', '#E69422', '#CD3E4E'])
    ax.set_ylabel('test AUC'); ax.set_ylim(0, 1.02); ax.set_title('Temporal ablations')
    for i, v in enumerate([ab['auc_full'], ab['auc_zero_dt'], ab['auc_shuffled']]):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    plt.show(); SUMMARY['figures'] += [str(p) for p in ce.save_fig(fig, OUT_DIR, 'extra_temporal_ablation')]
    SUMMARY['temporal_ablation'] = ab

## Save explain summary

In [ ]:
def _jsonable(o):
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

with open(OUT_DIR / 'explain_summary.json', 'w') as f:
    json.dump(SUMMARY, f, indent=2, default=_jsonable)
print('Saved', OUT_DIR / 'explain_summary.json', 'with', len(SUMMARY['figures']), 'figures.')
try:
    tracking.finish_run(WANDB_RUN)
except Exception:
    pass